In [ ]:
import pandas as pd
import mediapipe as mp
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score


In [ ]:
# Load dataset
file_path = "keypoint.csv"
df = pd.read_csv(file_path)

# Rename columns
num_features = df.shape[1]
df.columns = [f"feature_{i}" for i in range(num_features)]

In [ ]:
# Ensure consistent feature count
expected_features = 42  # 21 landmarks * 2 (x, y) if z is missing
actual_features = df.shape[1] - 1  # Excluding label column

if actual_features < expected_features:
    raise ValueError(f"Dataset contains {actual_features} features, but expected at least {expected_features}.")
    
X = df.iloc[:, 1:expected_features+1]  # Ensure exact match
# Target label
y = df.iloc[:, 0]

In [ ]:
# Split dataset (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train SVM model
svm_model = SVC(kernel='linear')
svm_model.fit(X_train, y_train)

# Predict on test set
y_pred = svm_model.predict(X_test)

# Evaluate model accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

In [ ]:
# Initialize MediaPipe Hands
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(min_detection_confidence=0.5, min_tracking_confidence=0.5)


In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

# Open webcam
cap = cv2.VideoCapture(0)
hands = mp_hands.Hands(min_detection_confidence=0.5, min_tracking_confidence=0.5, max_num_hands=12)

keypoints_history = {"Right Hand": [], "Left Hand": []}

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # Convert image to RGB
    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(image)
    
    # Extract keypoints if hand is detected
    if results.multi_hand_landmarks:
        for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
            keypoints = []
            x_min, y_min = float('inf'), float('inf')
            x_max, y_max = float('-inf'), float('-inf')
            
            for idx, landmark in enumerate(hand_landmarks.landmark):
                x, y = int(landmark.x * frame.shape[1]), int(landmark.y * frame.shape[0])
                keypoints.append(landmark.x)
                keypoints.append(landmark.y)
                
                # Update bounding box coordinates
                x_min = min(x_min, x)
                y_min = min(y_min, y)
                x_max = max(x_max, x)
                y_max = max(y_max, y)
                
                # Draw joint index number
                cv2.putText(frame, str(idx), (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
            
            # Ensure keypoints match expected feature size
            if len(keypoints) != 42:  # Pastikan jumlah keypoint sesuai
                print(f"Skipping frame: Expected 42 keypoints, got {len(keypoints)}")
                continue
            
            # Convert keypoints to NumPy array and reshape
            keypoints = np.array(keypoints).reshape(1, -1)
            
            # Tentukan apakah tangan kiri atau kanan
            hand_label = handedness.classification[0].label
            hand_text = "Right Hand" if hand_label == "Left" else "Left Hand"
            
            # Simpan keypoints untuk plotting
            keypoints_history[hand_text].append(keypoints.flatten().tolist())
            
            # Gambar landmark tangan
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            
            # Gambar bounding box
            cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
            
            # Tambahkan label tangan kanan atau kiri
            cv2.putText(frame, hand_text, (x_min, y_max + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            
            # Cetak keypoints ke terminal
            print(f"{hand_text} Keypoints: {keypoints.flatten().tolist()}")
    
    # Show frame
    cv2.imshow("Hand Recognition", frame)
    
    # Exit if 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
